In [ ]:
# ==========================================
# CELL 1: SETUP API & PATH FILE
# ==========================================
import os
import time
import pandas as pd
from groq import Groq

# Setup API Key Groq
api_key_groq = os.getenv("GROQ_API_KEY")

# Cek apakah terbaca (Print 8 karakter pertama saja untuk keamanan)
print(f"🔑 Key yang terbaca oleh sistem: {api_key_groq[:8]}...") 

# Jika hasil print memunculkan 'gsk_1yqR...' (tanpa tanda kutip), berarti SUKSES!
client = Groq(api_key=api_key_groq)
print("✅ Client Groq berhasil diinisialisasi!")

# Setup Path (Sesuaikan dengan struktur folder repo exigen-smart-maintenance Anda)
path_df_aset = "../../../data/master_aset_enriched.xlsx" 
path_df_perbaikan = "../../../data/aset_komplain_enriched.xlsx"

# ⬇️ PERUBAHAN: Nama file diganti agar tidak menimpa format kolom yang lama
path_output = "../../../data/dataset_tiket_lengkap.csv"

print("✅ Cell 1 Selesai: Library ter-import dan API Key sudah diset.")

🔑 Key yang terbaca oleh sistem: gsk_1yqR...
✅ Client Groq berhasil diinisialisasi!
✅ Cell 1 Selesai: Library ter-import dan API Key sudah diset.


In [2]:
# ==========================================
# CELL 2: LOAD DATA & AMBIL ATURAN (CONSTRAINTS)
# ==========================================
print("Membaca dataset asli NTG...")
df_aset = pd.read_excel(path_df_aset)
df_perbaikan = pd.read_excel(path_df_perbaikan)

# Mengambil daftar Kunci Jawaban (Target Y) agar AI tidak ngarang
list_severity = df_perbaikan['Severity'].dropna().unique().tolist()
list_penyebab = df_perbaikan['Penyebab'].dropna().unique().tolist()
list_jenis_kerusakan = df_perbaikan['Jenis Kerusakan'].dropna().unique().tolist() 

# Mengambil Kombinasi Aset Nyata (Feature X)
kombinasi_valid = df_aset[['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 
                           'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']].dropna().drop_duplicates()

# UNTUK TESTING: Kita ambil 3 kombinasi acak saja dulu agar prosesnya cuma hitungan detik
kombinasi_sample = kombinasi_valid.sample(3) 

print(f"✅ Cell 2 Selesai: Berhasil mengekstrak {len(kombinasi_sample)} kombinasi mesin untuk di-testing.")
print(f"🔸 Batas Severity yang diizinkan: {list_severity[:3]}...")

Membaca dataset asli NTG...
✅ Cell 2 Selesai: Berhasil mengekstrak 3 kombinasi mesin untuk di-testing.
🔸 Batas Severity yang diizinkan: ['Fatal', 'Ringan', 'Sedang']...


In [3]:
# ==========================================
# CELL 2.1: VALIDASI DATA UNIK (MAKESURE DATA BENAR)
# ==========================================

print("🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍\n")

# 1. Validasi Kolom Target (dari df_perbaikan)
print("--- [TARGET Y / KUNCI JAWABAN] ---")
print(f"🔹 Severity ({len(list_severity)}): {list_severity}")
print(f"🔹 Penyebab ({len(list_penyebab)}): {list_penyebab[:10]}... (dan seterusnya)")
print(f"🔹 Tindakan ({len(list_jenis_kerusakan)}): {list_jenis_kerusakan[:10]}... (dan seterusnya)")

# 2. Validasi Identitas & Lokasi (dari df_aset)
print("\n--- [IDENTITAS ASET & LOKASI] ---")
cols_aset = ['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']

for col in cols_aset:
    unique_vals = df_aset[col].dropna().unique().tolist()
    print(f"📍 {col} ({len(unique_vals)} unik): {unique_vals[:15]}") # Tampilkan 15 contoh pertama

# 3. Validasi Kombinasi yang Terpilih untuk Testing
print("\n--- [CONTOH KOMBINASI UNTUK TESTING GROQ] ---")
display(kombinasi_sample)

🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍

--- [TARGET Y / KUNCI JAWABAN] ---
🔹 Severity (4): ['Fatal', 'Ringan', 'Sedang', 'Berat']
🔹 Penyebab (12): ['Kelembaban tinggi', 'Human error', 'Overload', 'Kurang perawatan', 'Tegangan tidak stabil', 'Faktor lingkungan', 'Debu/kotoran', 'Aus normal', 'Usia pakai', 'Kualitas material']... (dan seterusnya)
🔹 Tindakan (20): ['Retak/pecah', 'Overheat', 'Remote tidak berfungsi', 'Getaran berlebihan', 'Korsleting', 'Aus/abrasi', 'Tidak berfungsi total', 'Kebocoran', 'Sensor error', 'Aliran lemah']... (dan seterusnya)

--- [IDENTITAS ASET & LOKASI] ---
📍 Kategori (15 unik): ['Mechanical', 'Ventilasi Sistem', 'Electrical', 'Sistem Pemadam Kebakaran', 'Sistem Telekomunikasi Gedung', 'Sistem Proteksi Kebakaran Aktif', 'Security Sistem', 'Civil', 'Plumbing', 'Distribusi Air', 'Sistem Transportasi Gedung', 'Pencatatan Meter', 'Arsitektur', 'Sistem Energi', 'Latihan Balakar']
📍 Sub Kategori (51 unik): ['Tata Udara', 'Sistem Sirkulasi Udara', 'Contro

,Kategori,Sub Kategori,Tipe,Merek,Lokasi Gedung,Lokasi Lantai,Lokasi Zona
26735,Civil,Dinding Bangunan,Dinding Tembok/Mansonry,Import,Gedung Utama,18,Selatan
13010,Electrical,Panel Distribusi,MDB,Import,Gedung Utama,6,Tengah
45920,Plumbing,Sanitari Sistem,Kran Air,Onda,Gedung D,12,Barat


In [4]:
# ==========================================
# CELL 2.2: STRATIFIED SAMPLING (JAMINAN 100% COVERAGE)
# ==========================================

print("Membuat antrean kasus komprehensif agar tidak ada variasi yang terlewat...")

# 1. Ambil SEMUA variasi unik dari histori perbaikan (Dijamin 100% terwakili)
kasus_unik = df_perbaikan.drop_duplicates(
    subset=['Tipe', 'Severity']
).copy()

print(f"Total variasi kasus unik di histori: {len(kasus_unik)} kombinasi.")

# 2. Pasangkan setiap kasus unik dengan spesifikasi Aset Nyata (Merek, Lokasi, dll)
antrean_prompt = []

for _, row in kasus_unik.iterrows():
    kategori_kasus = row['Kategori']
    
    # Cari aset di df_aset yang Kategori-nya cocok dengan kasus ini
    aset_cocok = kombinasi_valid[kombinasi_valid['Kategori'] == kategori_kasus]
    
    if not aset_cocok.empty:
        # Ambil 1 aset acak yang relevan untuk dijadikan "aktor" dalam skenario ini
        aset_pilih = aset_cocok.sample(1).iloc[0]
        
        antrean_prompt.append({
            'Kategori': kategori_kasus,
            'Tipe': aset_pilih['Tipe'],
            'Merek': aset_pilih['Merek'],
            'Lokasi Gedung': aset_pilih['Lokasi Gedung'],
            'Lokasi Lantai': aset_pilih['Lokasi Lantai'],
            'Lokasi Zona': aset_pilih['Lokasi Zona'],
            'Severity': row['Severity'],
            'Penyebab': row['Penyebab'],
            'Jenis Kerusakan': row['Jenis Kerusakan'],
            'Biaya Perbaikan': row['Biaya Perbaikan']
        })

df_antrean = pd.DataFrame(antrean_prompt)
print(f"✅ Antrean Siap! Total data yang HARUS di-generate AI: {len(df_antrean)} baris.")
display(df_antrean.head())

Membuat antrean kasus komprehensif agar tidak ada variasi yang terlewat...
Total variasi kasus unik di histori: 857 kombinasi.
✅ Antrean Siap! Total data yang HARUS di-generate AI: 857 baris.


,Kategori,Tipe,Merek,Lokasi Gedung,Lokasi Lantai,Lokasi Zona,Severity,Penyebab,Jenis Kerusakan,Biaya Perbaikan
0,Mechanical,AC Split,Gree,Gedung B,1,Barat,Fatal,Kelembaban tinggi,Retak/pecah,18771000
1,Security Sistem,Kamera CCTV,Dahua,Gedung Utama,10,Barat,Ringan,Human error,Overheat,356000
2,Electrical,LVMDP,Import,Gedung Parkir,4,Barat,Sedang,Overload,Remote tidak berfungsi,1610000
3,Mechanical,AC Cassette,Daikin,Gedung Utama,18,Selatan,Ringan,Kurang perawatan,Overheat,122000
4,Security Sistem,Kamera CCTV,Dahua,Gedung B,5,Timur,Sedang,Tegangan tidak stabil,Getaran berlebihan,1760000


In [ ]:
# ==========================================
# CELL 3: GENERATE DATASET DENGAN FITUR "SUPER RESUME" & DATA-DRIVEN COST
# ==========================================
import time
import os
import pandas as pd
import numpy as np
import random

# Ganti dengan path ke file yang Anda inginkan
path_utama = "../../../data/dataset_tiket_lengkap_lokasi_revisi.csv" 
path_revisi = "../../../data/dataset_tiket_lengkap_dupe_revisi.csv"
path_data_asli = "../../../data/aset_komplain_enriched.xlsx" # Asumsi ada data master asli

# =========================================================
# 1. PERSIAPAN DATA ASLI UNTUK KORELASI BIAYA (DATA-DRIVEN)
# =========================================================
print("📥 Membaca data asli NTG untuk mempelajari standar harga...")
try:
    df_asli = pd.read_excel(path_data_asli)
    # Sesuaikan nama kolom dengan file aset_komplain_enriched.xlsx Anda
    stat_biaya = df_asli.groupby(['Tipe', 'Severity'])['Biaya Perbaikan'].agg(['mean', 'std']).to_dict('index')
    fallback_biaya_tipe = df_asli.groupby('Tipe')['Biaya Perbaikan'].mean().to_dict()
    fallback_biaya_global = df_asli['Biaya Perbaikan'].mean()
    print("✅ Kamus Statistik Harga berhasil dibuat!")
except Exception as e:
    print(f"⚠️ Gagal membaca data asli untuk harga. Error: {e}")
    print("⚠️ Beralih ke mode Matrix Multiplier (Harga Hardcode).")
    stat_biaya = {} # Kosongkan jika gagal

# =========================================================
# 2. BACA KEDUA FILE UNTUK CEK KOMBINASI YANG SUDAH SELESAI
# =========================================================
kombinasi_selesai = set()

def ekstrak_kombinasi(path_file, nama_file):
    if os.path.exists(path_file):
        try:
            df_lama = pd.read_csv(path_file, sep='|', on_bad_lines='skip')
            # Cek jika kolomnya tersedia
            if set(['kategori_aset', 'severity', 'root_cause', 'jenis_kerusakan']).issubset(df_lama.columns):
                for _, row in df_lama.iterrows():
                    kunci = f"{row['kategori_aset']}_{row['severity']}_{row['root_cause']}_{row['jenis_kerusakan']}"
                    kombinasi_selesai.add(kunci)
                print(f"✅ [{nama_file}] Terbaca. Menambahkan ke daftar antrean yang di-skip.")
        except Exception as e:
            print(f"⚠️ Gagal membaca {nama_file}. Error: {e}")

# Load kombinasi dari kedua file
ekstrak_kombinasi(path_utama, "File Utama")
ekstrak_kombinasi(path_revisi, "File Revisi")

print(f"🛡️ Total kombinasi unik yang sudah diamankan: {len(kombinasi_selesai)} kasus.")

# Buat file utama beserta headernya jika belum ada
if not os.path.exists(path_utama):
    print("📁 File utama belum ada. Membuat file baru beserta headernya...")
    header_kolom = "teks_keluhan_awam|teks_laporan_teknisi|tipe_aset|lokasi_gedung|lokasi_lantai|lokasi_zona|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan\n"
    with open(path_utama, 'w', encoding='utf-8') as f:
        f.write(header_kolom)

# =========================================================
# 3. FILTER ANTREAN (Hanya ambil yang belum ada di CSV)
# =========================================================
antrean_sisa = []
for index, row in df_antrean.iterrows():
    kunci_antrean = f"{row['Kategori']}_{row['Severity']}_{row['Penyebab']}_{row['Jenis Kerusakan']}"
    if kunci_antrean not in kombinasi_selesai:
        antrean_sisa.append((index, row))

print(f"🎯 Total variasi unik keseluruhan target: {len(df_antrean)}")
print(f"⏩ Sisa antrean murni yang BELUM dikerjakan AI: {len(antrean_sisa)}\n")


# =========================================================
# 4. PROSES LOOPING API GROQ
# =========================================================
csv_batch_baru = "" # Variabel penampung hasil baru

if len(antrean_sisa) > 0:
    print(f"🚀 Mulai memanggil LLaMA 3 untuk sisa antrean...")
    for index_asli, row in antrean_sisa:
        kat, tipe, merek = row['Kategori'], row['Tipe'], row['Merek']
        ged, lan, zon = row['Lokasi Gedung'], row['Lokasi Lantai'], row['Lokasi Zona']
        sev, rc, jk = row['Severity'], row['Penyebab'], row['Jenis Kerusakan']
        
        # --- LOGIC PENENTUAN BIAYA (PYTHON YANG HITUNG, BUKAN AI) ---
        kunci_biaya = (tipe, sev)
        if stat_biaya and kunci_biaya in stat_biaya and not pd.isna(stat_biaya[kunci_biaya]['mean']):
            mean_asli = stat_biaya[kunci_biaya]['mean']
            std_asli = stat_biaya[kunci_biaya]['std']
            if pd.isna(std_asli) or std_asli == 0:
                std_asli = mean_asli * 0.15 
            biaya_aktual = np.random.normal(loc=mean_asli, scale=std_asli)
        elif stat_biaya and tipe in fallback_biaya_tipe:
            biaya_aktual = fallback_biaya_tipe[tipe] * random.uniform(0.8, 1.2)
        elif stat_biaya:
            biaya_aktual = fallback_biaya_global * random.uniform(0.8, 1.2)
        else:
            # FALLBACK HARDCODE Jika file Excel tidak ada
            harga_dasar = 500000
            if sev == 'Ringan': biaya_aktual = (harga_dasar * 0.3) * random.uniform(0.8, 1.2)
            elif sev == 'Sedang': biaya_aktual = (harga_dasar * 1.0) * random.uniform(0.8, 1.2)
            elif sev == 'Berat': biaya_aktual = (harga_dasar * 3.0) * random.uniform(0.8, 1.2)
            else: biaya_aktual = (harga_dasar * 8.0) * random.uniform(0.8, 1.2)
            
        biaya_aktual = max(biaya_aktual, 50000) # Minimal 50rb
        biaya = int(round(biaya_aktual, -4))    # Bulatkan ke puluhan ribu
        # -------------------------------------------------------------
        
        print(f"🔄 Memproses [Antrean {index_asli+1}/{len(df_antrean)}]: {kat} - {jk} | Biaya Set: Rp {biaya:,}...")
        
        # 🧠 THE ULTIMATE PROMPT V2
        prompt = f"""
        Anda adalah AI Senior Data Scientist spesialis pembuatan dataset NLP (Natural Language Processing).
        TUGAS ANDA HANYA MENGELUARKAN TEKS CSV MURNI TANPA HEADER, TANPA BASA-BASI, TANPA BLOK MARKDOWN (```csv).

        Buatkan TEPAT 3 baris data CSV untuk skenario perbaikan historis berikut:
        - Tipe Aset: {tipe}
        - Kategori: {kat}
        - Merek: {merek}
        - Lokasi: Gedung {ged}, Lantai {lan}, Zona {zon}
        - Severity: {sev}
        - Root Cause: {rc}
        - Jenis Kerusakan: {jk}
        - Biaya Perbaikan Aktual: Rp {biaya}

        🔥 ATURAN 1: WAJIB SEBUT LOKASI LENGKAP DI KELUHAN (HARGA MATI)
        Teks 'teks_keluhan_awam' WAJIB SEKALI menyisipkan informasi (Gedung {ged}, Lantai {lan}, Zona {zon}) secara eksplisit dan natural. 
        Contoh SALAH: "Tolong dong {tipe}-nya rusak."
        Contoh BENAR: "Tolong dong {tipe} di Gedung {ged} lantai {lan} zona {zon} rusak nih."

        🔥 ATURAN 2: KOSAKATA KEPARAHAN (SEVERITY NOISE REDUCTION)
        DILARANG KERAS menggunakan kata sifat yang sama untuk kelas severity berbeda. Sesuaikan 'teks_keluhan_awam' dengan status '{sev}':
        - Fatal: meledak, terbakar, ambruk, bocor deras, banjir, nyetrum, bau gosong. (Nada: SANGAT PANIK)
        - Berat: mati total, mogok, patah, tidak nyala sama sekali, jebol. (Nada: MENDESAK)
        - Sedang: netes air, berisik banget, kedap-kedip, kurang dingin, error terus. (Nada: TERGANGGU)
        - Ringan: kotor, berdebu, kusam, bunyi pelan, tombol agak keras. (Nada: SANTAI)

        🔥 ATURAN 3: GAYA BAHASA KELUHAN AWAM (ANTI-TEMPLATE)
        Ketiga baris WAJIB 100% berbeda strukturnya.
        - Baris 1: Bahasa WA keseharian, panjang, pakai kata seru dan singkatan (yg, dmn, bgt, gk).
        - Baris 2: Sangat singkat, padat, gaya telegram cepat (misal: "Lapor. {tipe} lt {lan} {zon} mati.").
        - Baris 3: Bahasa staf kantoran, sopan dan terstruktur.

        🔥 ATURAN 4: ATURAN TEKS LAPORAN TEKNISI & BIAYA (SANGAT KETAT)
        - 'teks_laporan_teknisi' WAJIB menggunakan bahasa teknis/engineering yang menyebutkan '{rc}' dan tindakan perbaikan.
        - LARANGAN KERAS: Pelapor (awam) TIDAK TAHU biaya perbaikan! JANGAN PERNAH memasukkan angka nominal Rp {biaya} di dalam 'teks_keluhan_awam'.
        - Nominal Rp {biaya} HANYA BOLEH disebutkan di dalam 'teks_laporan_teknisi' (opsional) DAN diletakkan di kolom paling akhir.

        🔥 ATURAN 5: FORMATTING CSV STRICT
        1. DILARANG KERAS menggunakan simbol '|' (pipa) atau 'Enter/Newline' di dalam kalimat keluhan maupun laporan teknisi!
        2. Jangan ada baris kosong di antara ke-3 baris output.
        3. Pastikan tepat ada 10 pipa ('|') di setiap baris (menghasilkan 11 kolom).

        Format Output WAJIB (Tanpa spasi setelah pipa):
        teks_keluhan_awam|teks_laporan_teknisi|{tipe}|{ged}|{lan}|{zon}|{kat}|{sev}|{rc}|{jk}|{biaya}
        """
        
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile", 
                messages=[
                    {"role": "system", "content": "You output ONLY raw valid CSV text without headers."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.8, 
                max_tokens=2000
            )
            
            hasil_text = response.choices[0].message.content.strip()
            hasil_text = hasil_text.replace("```csv", "").replace("```", "").strip()
            
            if hasil_text.lower().startswith("teks_keluhan"):
                hasil_text = "\n".join(hasil_text.split("\n")[1:])
                
            csv_batch_baru += hasil_text + "\n" 
            
            # AUTO-SAVE LANGSUNG KE FILE UTAMA
            with open(path_utama, 'a', encoding='utf-8') as f:
                f.write(hasil_text + "\n")
                
            time.sleep(10) # Jeda untuk Groq API
            
        except Exception as e:
            print(f"❌ Terkena Limit API pada baris {index_asli+1}. Looping dihentikan aman. Error: {e}")
            break 

    print("✅ Cell 3 Selesai! Data aman tersimpan di CSV.")
else:
    print("🎉 Hore! Semua variasi kasus sudah selesai di-generate. Tidak perlu memanggil API lagi.")

📥 Membaca data asli NTG untuk mempelajari standar harga...
✅ Kamus Statistik Harga berhasil dibuat!
✅ [File Utama] Terbaca. Menambahkan ke daftar antrean yang di-skip.
✅ [File Revisi] Terbaca. Menambahkan ke daftar antrean yang di-skip.
🛡️ Total kombinasi unik yang sudah diamankan: 536 kasus.
🎯 Total variasi unik keseluruhan target: 857
⏩ Sisa antrean murni yang BELUM dikerjakan AI: 284

🚀 Mulai memanggil LLaMA 3 untuk sisa antrean...
🔄 Memproses [Antrean 562/857]: Electrical - Remote tidak berfungsi | Biaya Set: Rp 220,000...
🔄 Memproses [Antrean 563/857]: Mechanical - Tekanan tidak stabil | Biaya Set: Rp 5,190,000...
🔄 Memproses [Antrean 564/857]: Latihan Balakar - Tampilan rusak | Biaya Set: Rp 6,400,000...
🔄 Memproses [Antrean 565/857]: Electrical - Aus/abrasi | Biaya Set: Rp 24,430,000...
🔄 Memproses [Antrean 566/857]: Civil - Tampilan rusak | Biaya Set: Rp 990,000...
🔄 Memproses [Antrean 567/857]: Distribusi Air - Lampu mati | Biaya Set: Rp 3,150,000...
🔄 Memproses [Antrean 568/8

In [6]:
# ==========================================
# CELL 4: SIMPAN & BACA HASIL (APPEND MODE)
# ==========================================

# 1. Simpan Data Baru (Jika ada)
if 'csv_batch_baru' in locals() and csv_batch_baru.strip() != "":
    
    # Jika file belum pernah ada sama sekali, buatkan sekalian beserta headernya
    if not os.path.exists(path_output):
        with open(path_output, "w", encoding="utf-8") as f:
            # ⬇️ PERUBAHAN: Header disesuaikan menjadi 11 Kolom
            f.write("teks_keluhan_awam|teks_laporan_teknisi|tipe_aset|lokasi_gedung|lokasi_lantai|lokasi_zona|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan\n")
            
    # Tambahkan (Append) data baru ke baris paling bawah file CSV
    with open(path_output, "a", encoding="utf-8") as f:
        f.write(csv_batch_baru)
        
    print(f"💾 File berhasil di-update dan disimpan di: {path_output}")
    
    # KOSONGKAN variabel setelah disimpan agar tidak ter-save ganda jika Anda me-run Cell 4 dua kali
    csv_batch_baru = "" 
else:
    print("ℹ️ Tidak ada teks baru untuk ditambahkan ke file saat ini.")

# 2. Coba baca keseluruhan dataset menggunakan Pandas
try:
    df_hasil = pd.read_csv(path_output, sep='|', on_bad_lines='skip')
    print(f"\n📊 DATASET FINAL! Total Keseluruhan Data Saat Ini: {df_hasil.shape[0]} baris.")
    display(df_hasil.head())
    display(df_hasil.tail()) # Tampilkan juga bagian paling bawah untuk ngecek
except Exception as e:
    print(f"❌ Gagal membaca CSV. Error: {e}")

💾 File berhasil di-update dan disimpan di: ../../data/dataset_tiket_lengkap.csv

📊 DATASET FINAL! Total Keseluruhan Data Saat Ini: 1203 baris.


,teks_keluhan_awam,teks_laporan_teknisi,tipe_aset,lokasi_gedung,lokasi_lantai,lokasi_zona,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
0,Tolong dong APAR CO2 di Gedung Gedung Utama la...,"APAR CO2 mengalami aus normal, perlu dilakukan...",APAR CO2,Gedung Utama,10,Tengah,Sistem Pemadam Kebakaran,Berat,Aus normal,Aus/abrasi,2770000.0
1,Lapor. APAR CO2 lt 10 Tengah mati.,"APAR CO2 mengalami kerusakan berat, perlu dila...",APAR CO2,Gedung Utama,10,Tengah,Sistem Pemadam Kebakaran,Berat,Aus normal,Aus/abrasi,2770000.0
2,Mohon bantuan untuk memperbaiki APAR CO2 di Ge...,"APAR CO2 mengalami aus normal, perlu dilakukan...",APAR CO2,Gedung Utama,10,Tengah,Sistem Pemadam Kebakaran,Berat,Aus normal,Aus/abrasi,2770000.0
3,Tolong dong Urinal di Gedung Gedung E lantai 6...,"Kerusakan disebabkan kurang perawatan, perlu d...",Urinal,Gedung E,6,Barat,Plumbing,Sedang,Kurang perawatan,Kebocoran,1930000.0
4,"Lapor, Urinal Gedung E lantai 6 zona Barat ber...","Kerusakan disebabkan kurang perawatan, dilakuk...",Urinal,Gedung E,6,Barat,Plumbing,Sedang,Kurang perawatan,Kebocoran,1930000.0


,teks_keluhan_awam,teks_laporan_teknisi,tipe_aset,lokasi_gedung,lokasi_lantai,lokasi_zona,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
1198,"Lapor, Lantai Keramik lt 6 Timur rusak",Kerusakan lantai keramik disebabkan oleh kuran...,Lantai Keramik,Gedung B,6,Timur,Civil,Ringan,Kurang perawatan,Tampilan rusak,390000.0
1199,Mohon bantuan untuk memperbaiki Lantai Keramik...,Lantai Keramik mengalami kerusakan akibat kura...,Lantai Keramik,Gedung B,6,Timur,Civil,Ringan,Kurang perawatan,Tampilan rusak,390000.0
1200,Tolong dong Atap Genteng di Gedung Gedung Utam...,"Kerusakan disebabkan oleh Korosi, perlu dilaku...",Atap Genteng,Gedung Utama,18,Utara,Civil,Berat,Korosi,Getaran berlebihan,7080000.0
1201,"Lapor, Atap Genteng lt 18 Utara mati total!","Kerusakan akibat Korosi, perlu perbaikan segera",Atap Genteng,Gedung Utama,18,Utara,Civil,Berat,Korosi,Getaran berlebihan,7080000.0
1202,"Mohon perhatian, Atap Genteng di Gedung Gedung...","Kerusakan disebabkan oleh Korosi, dilakukan pe...",Atap Genteng,Gedung Utama,18,Utara,Civil,Berat,Korosi,Getaran berlebihan,7080000.0
